# SRP-conditions-predict-using-ML - Colab runner

**This notebook is deliberately thin. It contains NO experiment logic.**
Everything scientific lives in the repository, under version control. This notebook
only: mounts Drive, clones, installs, points DATA_ROOT at the dataset in Drive,
symlinks `artifacts/` into Drive, calls ONE script, and zips `artifacts/` for
committing.

If you find yourself editing an experiment parameter here, stop: edit `configs/*.yaml`
in the repository, push, and re-run this notebook.

See `docs/COLAB_SETUP.md` for the Drive layout and how to upload the dataset.

---
## Free-tier realities

Free Colab is not a batch queue. It will interrupt you, and the notebook is built
around that rather than pretending otherwise.

| what | what it means here |
| --- | --- |
| **~90 min idle disconnect** | the runtime is reclaimed if the browser tab stops talking to it. **Keep the tab open and the machine awake while a script runs.** |
| **dynamic session length** | there is no guaranteed 12 h. A session can end at any point, with no warning and no way to appeal. |
| **dynamic GPU quota** | after heavy use, Colab silently hands you a **CPU-only** runtime for hours. Cell 3 checks for this loudly, because a script that starts on CPU is ~an order of magnitude slower and you want to know before, not after. |

**So: split the long scripts across sessions and across days.** `03_run_cv.py` is 75
runs; do not expect it in one sitting. Start it, let it run as long as the session
lasts, and start it again next time — it skips everything already in the registry
and picks up where it stopped. The same is true of every script here.

---
## How results survive a dropped session

`artifacts/` in the repository is **symlinked into Drive** (cell 5). Everything the
scripts write — `registry.jsonl` above all — lands in Drive *as it is written*, not
at the end. The registry is flushed and fsync'd after every completed run, so a
session that dies mid-run loses only that run.

**A dropped session therefore costs nothing.** Reconnect, re-run cells 1-5, and
re-run the script; it resumes.

This is the opposite of the Kaggle runner, where nothing survives unless you
explicitly copy `artifacts/` out and commit it before the session ends. Here the
live copy is already safe in Drive, and the zip at the end is for **git versioning
only**.


## 0. Settings

The only variables in this notebook. Everything else is derived.


In [ ]:
# Where the dataset lives in your Drive. Must contain the `dataset/` wrapper
# directory, which in turn holds the 10 class directories. See docs/COLAB_SETUP.md.
DATA_ROOT_DRIVE = '/content/drive/MyDrive/srp-dyna-card/dataset'

# Where artifacts/ is redirected to. Created if absent. This is what makes a
# dropped session survivable, so it must be in Drive, not in /content.
ARTIFACTS_DRIVE = '/content/drive/MyDrive/srp-artifacts'

REPO_URL = 'https://github.com/akiraraihaan/SRP-conditions-predict-using-ML.git'
BRANCH   = 'main'
WORK     = '/content/SRP-conditions-predict-using-ML'

print('dataset   :', DATA_ROOT_DRIVE)
print('artifacts :', ARTIFACTS_DRIVE)
print('repo      :', REPO_URL, '@', BRANCH)


## 1. Mount Google Drive

Authorise when prompted. Drive holds both the dataset and the live `artifacts/`,
so nothing below works until this succeeds.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.isdir('/content/drive/MyDrive'):
    raise SystemExit('Drive did not mount: /content/drive/MyDrive is not a directory')
print('Drive mounted.')


## 2. Clone the repository

A fresh clone every session. Nothing is edited in place here — edit the repo,
push, re-run.


In [ ]:
import os, shutil, subprocess

if os.path.exists(WORK):
    shutil.rmtree(WORK)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, WORK], check=True)
os.chdir(WORK)
print('cwd:', os.getcwd())
subprocess.run(['git', 'log', '-1', '--oneline'], check=False)


## 3. Install dependencies, and check what runtime you actually got

Colab's image already carries a CUDA build of torch. Let the pre-installed wheel
win rather than forcing a reinstall of the CPU pin from `requirements.txt`.

**Read the output of this cell before going further.** Free Colab silently gives you
a CPU-only runtime once your GPU quota is exhausted, and it does not tell you. The
scripts will run on CPU perfectly correctly and roughly an order of magnitude
slower, which is the kind of thing you want to discover now rather than six hours in.


In [ ]:
!pip install -q --upgrade-strategy only-if-needed -r requirements.txt

import torch
print('torch      :', torch.__version__)
print('torch.cuda :', torch.version.cuda)
print('available  :', torch.cuda.is_available())

if torch.cuda.is_available():
    print('gpu        :', torch.cuda.get_device_name(0))
    print('memory     : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
else:
    bar = '!' * 74
    print()
    print(bar)
    print('CPU-ONLY RUNTIME -- NO GPU IS ATTACHED')
    print()
    print('  Either the runtime type is wrong, or your free-tier GPU quota is spent.')
    print('  1. Runtime -> Change runtime type -> Hardware accelerator: GPU -> Save.')
    print('     If that reconnects with a GPU, re-run cells 1-5 and carry on.')
    print('  2. If it still says None, or GPU is greyed out, the quota is exhausted.')
    print('     It comes back on its own, typically after several hours to a day.')
    print()
    print('  Training will still run, correctly, but ~10x slower. A 75-run script is')
    print('  not realistic on CPU. Wait for the quota rather than starting it.')
    print(bar)


## 4. Point DATA_ROOT at the dataset in Drive

Nothing is hardcoded: the path comes from `configs/data.yaml` and is overridden here
by the environment variable, so the same code runs unchanged on Kaggle, locally and
on a Raspberry Pi.

This cell **fails loudly, printing the path it resolved**, if the 10 class
directories are not where it expects. That is almost always a Drive layout problem —
see `docs/COLAB_SETUP.md`.


In [ ]:
import os, yaml

os.environ['SRPCARD_DATA_ROOT'] = DATA_ROOT_DRIVE

expected = yaml.safe_load(open('configs/data.yaml', encoding='utf-8'))['classes']

if not os.path.isdir(DATA_ROOT_DRIVE):
    raise SystemExit(
        'DATA_ROOT does not exist or is not a directory:\n'
        '  resolved to : %s\n'
        '  Check DATA_ROOT_DRIVE in cell 0, and that Drive is mounted.\n'
        '  The path must include the dataset/ wrapper directory.\n'
        '  See docs/COLAB_SETUP.md.' % DATA_ROOT_DRIVE
    )

found = sorted(d for d in os.listdir(DATA_ROOT_DRIVE)
               if os.path.isdir(os.path.join(DATA_ROOT_DRIVE, d)))
missing = [c for c in expected if c not in found]
unexpected = [d for d in found if d not in expected]

print('DATA_ROOT resolved to :', DATA_ROOT_DRIVE)
print('class directories     : %d found, %d expected' % (len(found), len(expected)))

if missing or unexpected:
    print()
    print('  found      :', found)
    print('  missing    :', missing)
    print('  unexpected :', unexpected)
    raise SystemExit(
        'DATA_ROOT does not hold the 10 canonical class directories.\n'
        '  resolved to : %s\n'
        '  A common cause is pointing at the parent instead of the dataset/ wrapper,\n'
        '  or an interrupted Drive upload. See docs/COLAB_SETUP.md.' % DATA_ROOT_DRIVE
    )

total = sum(len(files) for _, _, files in os.walk(DATA_ROOT_DRIVE))
print('files under DATA_ROOT  :', total, '(695 expected)')
print()
for name in expected:
    n = len(os.listdir(os.path.join(DATA_ROOT_DRIVE, name)))
    print('  %-45s %4d' % (name, n))


## 5. Redirect `artifacts/` into Drive

**This is the cell that makes a dropped session cost nothing.** `artifacts/` in the
clone is replaced by a symlink to `ARTIFACTS_DRIVE`, so every file the scripts write
lands in Drive at the moment it is written — the registry, the summary tables, the
figures.

Two directions of copying, and they are not symmetric:

- **Frozen inputs → Drive.** `image_index.csv`, `folds.json`, `dev_split.json` and
  the other committed artefacts are refreshed from the clone every session. The
  repository is authoritative for those; no script writes them.
- **`registry.jsonl` ← Drive, never the other way.** The committed copy is empty.
  Overwriting Drive's with it would erase every completed run. It is copied from the
  clone *only* if Drive does not already have one.


In [ ]:
import os, shutil

FROZEN = [
    'image_index.csv', 'folds.json', 'dev_split.json', 'excluded_images.csv',
    'folds_report.md', 'legacy_contamination.json', 'legacy_grid_metrics.csv',
    'legacy_test_predictions.csv',
]
PRESERVE = ['registry.jsonl']   # Drive is authoritative; NEVER clobbered by the clone

os.makedirs(ARTIFACTS_DRIVE, exist_ok=True)
repo_artifacts = os.path.join(WORK, 'artifacts')

# frozen inputs: the clone wins
for name in FROZEN:
    src = os.path.join(repo_artifacts, name)
    if os.path.exists(src) and not os.path.islink(repo_artifacts):
        shutil.copy2(src, os.path.join(ARTIFACTS_DRIVE, name))

# resume state: Drive wins, and is only seeded if absent
for name in PRESERVE:
    dst = os.path.join(ARTIFACTS_DRIVE, name)
    if not os.path.exists(dst):
        src = os.path.join(repo_artifacts, name)
        if os.path.exists(src) and not os.path.islink(repo_artifacts):
            shutil.copy2(src, dst)
        else:
            open(dst, 'a').close()
        print('seeded a new %s in Drive' % name)

# replace artifacts/ with the symlink
if os.path.islink(repo_artifacts):
    os.unlink(repo_artifacts)
elif os.path.isdir(repo_artifacts):
    shutil.rmtree(repo_artifacts)
os.symlink(ARTIFACTS_DRIVE, repo_artifacts)

print('artifacts/ ->', os.path.realpath(repo_artifacts))
assert os.path.isdir(repo_artifacts), 'symlink did not resolve to a directory'

reg = os.path.join(repo_artifacts, 'registry.jsonl')
n = sum(1 for line in open(reg, encoding='utf-8') if line.strip()) if os.path.exists(reg) else 0
print('registry.jsonl holds %d completed run(s)' % n)
print()
for f in sorted(os.listdir(repo_artifacts)):
    p = os.path.join(repo_artifacts, f)
    if os.path.isfile(p):
        print('  %9.1f KB  %s' % (os.path.getsize(p) / 1024, f))


## 6. Run ONE script

Uncomment exactly one line. Run them in this order across sessions:

| script | what | runs |
|---|---|---|
| `00_build_folds.py` | **preflight**, then index, dev split, folds. Verifies committed artefacts. | - |
| `01_complete_medium_grid.py` | 8 missing medium configs, legacy protocol | 8 |
| `02_lr_sweep_baselines.py` | baseline lr sweep | 6 |
| `03_run_cv.py` | the main experiment | 75 |
| `04_run_ablation.py` | class-weight ablation | 15 |
| `05_learning_curve.py` | learning curve, 5 fractions x 15 folds | 75 |
| `06_export_figures.py` | figures and tables | - |

**Run `00_build_folds.py` first in every fresh session.** Its phase 0 preflight
reports the GPU and torch/CUDA versions, whether `cudnn.deterministic` took effect,
the resolved DATA_ROOT with the per-class counts it actually found, whether the
committed artefacts still match their fingerprints, and whether the pretrained
checkpoint of all five arms downloads. It exits non-zero if any of that failed.
`--preflight-only` runs just that part.

Scripts 01 and 02 write back into `configs/arms.yaml`. That file lives in the clone,
**not** in Drive, so it dies with the session — download it and commit it as soon as
those scripts finish.

If you lose it, the next session does **not** silently train the stale
hyperparameters. `epochs`, `batch` and `lr` feed the `run_id` hash, so a reverted
config would not resume — it would retrain every fold under the old settings and
leave two regimes in the registry. Scripts 03–05 detect that and **abort**, and
`00_build_folds.py`'s preflight reports it before anything runs. Both scripts also
snapshot the resolved config to `artifacts/resolved_arms.yaml`, which *is* in Drive,
so you can put it back:

```
!python scripts/restore_arms.py --check   # show what differs
!python scripts/restore_arms.py           # restore it, then commit arms.yaml
```

Every script is resumable: re-running skips what is already in the registry, which
is what makes the 75-run scripts practical across several short free-tier sessions.
Expect to run `03_run_cv.py` on more than one day.

**On `--allow-pretrained-fallback`:** `configs/arms.yaml` names a YOLO11 checkpoint
as each YOLO arm's `pretrained_fallback`. It is never taken automatically — a run
that trained YOLO11 while every table said YOLO26 would be undetectable afterwards.
If the YOLO26 download fails the script refuses and names both architectures. Pass
`--allow-pretrained-fallback` only if you have decided to accept the substitution;
it prints a loud banner and flags `pretrained_fallback_used` on every affected
registry record. Do not pass it by reflex to get a session unstuck.


In [ ]:
!python scripts/00_build_folds.py
# !python scripts/01_complete_medium_grid.py
# !python scripts/02_lr_sweep_baselines.py
# !python scripts/03_run_cv.py --quiet
# !python scripts/04_run_ablation.py --quiet
# !python scripts/05_learning_curve.py --quiet
# !python scripts/06_export_figures.py


## 7. Zip `artifacts/` — for git versioning, not for survival

**This zip is not how your results survive.** Drive already holds the live copy;
cell 5 saw to that, and it was updated after every completed run. If the session
died before you reached this cell, nothing was lost.

What the zip is for is **committing to git**, so the repository carries the same
history as Drive: `registry.jsonl`, `resolved_arms.yaml`, the summary tables, the
figures, and `configs/arms.yaml` if scripts 01 or 02 rewrote it.

This is the opposite of the Kaggle runner's last cell, which you must run before the
session ends or lose the session's work.


In [ ]:
import shutil, os, glob

OUT = '/content/artifacts_out'
if os.path.exists(OUT):
    shutil.rmtree(OUT)
os.makedirs(OUT)

for src in glob.glob('artifacts/**/*', recursive=True):
    if os.path.isfile(src):
        dst = os.path.join(OUT, os.path.relpath(src, 'artifacts'))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)

shutil.copy2('configs/arms.yaml', os.path.join(OUT, 'arms.yaml'))

for root, _, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f)
        print('%9.1f KB  %s' % (os.path.getsize(p) / 1024, os.path.relpath(p, OUT)))

shutil.make_archive('/content/artifacts_bundle', 'zip', OUT)
print()
print('bundle: /content/artifacts_bundle.zip  (Files pane -> download)')
print('A copy also goes to Drive so it survives the session:')
shutil.copy2('/content/artifacts_bundle.zip', ARTIFACTS_DRIVE + '/../artifacts_bundle.zip')
print('  ' + os.path.realpath(ARTIFACTS_DRIVE + '/../artifacts_bundle.zip'))
print()
print('Commit artifacts/registry.jsonl, artifacts/resolved_arms.yaml and any')
print('changed configs/arms.yaml.')
print('See HANDOVER.md section 4.7 for the full list.')


## 8. Progress check

How much is done, how much is left, and whether anything in the registry is
suspect.


In [ ]:
import json, collections, sys

counts = collections.Counter()
fallback = collections.Counter()
corpora = collections.Counter()
unverified = 0
records = []
try:
    with open('artifacts/registry.jsonl', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                records.append(json.loads(line))
except FileNotFoundError:
    print('no registry yet')

for r in records:
    counts[(r.get('script'), r.get('arm'))] += 1
    if r.get('pretrained_fallback_used'):
        fallback[(r.get('arm'), r.get('checkpoint_resolved'))] += 1
    if r.get('class_weights_verified') is False:
        unverified += 1
    fp = r.get('corpus_fingerprint') or {}
    corpora[(fp.get('kind'), fp.get('sha1_of_sorted_included_sha1s'))] += 1

for (script, arm), n in sorted(counts.items()):
    print('%-26s %-20s %3d' % (script, arm, n))
print('total records:', sum(counts.values()))

TARGET = {'01_complete_medium_grid': 8, '02_lr_sweep_baselines': 6,
          '03_run_cv': 75, '04_run_ablation': 15, '05_learning_curve': 75}
print()
by_script = collections.Counter(r.get('script') for r in records)
for script, target in sorted(TARGET.items()):
    done = by_script.get(script, 0)
    print('  %-26s %3d / %3d   %s' % (script, done, target,
                                      'done' if done >= target else '%d left' % (target - done)))

# Schema drift: a record missing a required field still matches by run_id, so its
# run is SKIPPED and its older numbers are inherited into the final results.
sys.path.insert(0, 'src')
from srpcard import registry
print()
registry.warn_if_stale()

print()
if fallback:
    print('*** %d record(s) used a PRETRAINED FALLBACK -- the architecture is not'
          ' what the arm declares:' % sum(fallback.values()))
    for (arm, ckpt), n in sorted(fallback.items()):
        print('      %-20s %-26s %3d run(s)' % (arm, ckpt, n))
    print('    These must not be reported as the declared architecture.')
else:
    print('no run used a pretrained fallback')

if unverified:
    print('*** %d record(s) have class_weights_verified=False' % unverified)
else:
    print('no run has a failed class-weight proof')

for (kind, sha1), n in sorted(corpora.items()):
    print('corpus %-16s %s  %3d run(s)' % (kind, sha1, n))
